In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78521)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 69173)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 57737)

# #### Device() ####
# device = cuda:6



In [2]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:6)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

---

* 3: ae (NBLoss only) vs vae (w/ KLDLoss)
    * similar to grid4a-c, using NBLoss and method=node

In [3]:
from modules.layers import MultiheadSetPooling
from modules.loss import MultiLoss, NBLoss, KLDLoss
from modules.model import CountsAutoencoder
from modules.norm import RawCounts, LogCounts, NBVST
from modules.train import Experiment, grid, kwarg_grid
from modules.trainers import ReconstrTrainer
from functools import partial
import torch.nn as nn

In [4]:
## Model

model_template = partial(
    CountsAutoencoder,
    dataset = dataset,
    # out_dim = num_nodes (default)
    # embed_dim = 128, !!! handled in grid !!!
    # head_dim = None (default)
    # num_heads = 1 (default) !!! handled in grid !!!
    method = 'node',

    # layers
    norm_class = LogCounts,
    encoder_class = nn.Linear,
    pooling_class = MultiheadSetPooling,
    mlp = False,
    variational = True,
    # out_module = nn.Linear (default)

    # layer params
    hidden_dims = 1,
    act_fn = nn.ReLU, 
    norm_fn = 'layer', 
    end_fn = False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    # pooling_kwargs = None (default) !!! handled in grid !!!
)

model_grid = grid(
    model_template,
    embed_dim = [64, 128, 256],
    num_heads = [1, 2, 4, 8],
    pooling_kwargs = [{'aggregate':'concat'}, {'aggregate':'mean'}]
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    lr = 1e-3,
    # pos_keys = None (default)
    kw_keys=('x','x_pred','theta','z_mu','z_logvar'),
    metric_keys={'pred':'x_pred', 'target':'x'},
    loss_class = MultiLoss,
    loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu', 'logvar':'z_logvar'}
        ]
    },
    # loss_kwargs = None (default)
    # optim_class = torch.optim.Adam (default)
    # optim_kwargs = None (default)
    early_stop = True,
    # stop_metric = 'loss', (default)
    # stop_kwargs = None (default)
    trainer_norm_class=LogCounts,
    # trainer_norm_kwargs = None (default)
)


trainer_grid = grid(
    trainer_template,
)

## Experiment

expt = Experiment(
    num_trials=5, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

24


In [5]:
expt.configs

{'embeddim64_numheads1_aggregateConcat': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads1_aggregateMean': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads2_aggregateConcat': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads2_aggregateMean': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads4_aggregateConcat': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads4_aggregateMean': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads8_aggregateConcat': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim64_numheads8_aggregateMean': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim128_numheads1_aggregateConcat': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim128_numheads1_aggregateMean': <modules.trainers.ReconstrTrainer at 0x7f36540b2960>,
 'embeddim128_numheads2_aggregateConcat': <modules.trainer

In [6]:
expt.run_experiment(
    'grid_5a_NBreconstr_multihead',
    report_metrics=['loss','kld','nb','mae'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]